# VERIFY — F1, F2, F4, F5, and the ResNet-18 swap

Run this **after** `00_Setup_and_Crypto_Tests.ipynb` has already passed once
in this session (same Drive mount, same clone). This notebook checks the
fixes logged in `FIX.md` as of commit `0bdd2da`:

| Fix | What to look for below |
|---|---|
| ResNet-18 swap | `model_dim=` near 11.2M, not 23,628,798 |
| F1 (unmasking) | `aggregate=EXACT` on **every** round line |
| F2 (AES mask PRG) | `phase1=` far below the old 1102.9s |
| F4 (batched sketch) | bootstrap prints one line per batch, not per sample; `phase2=` far below 257.7s |
| F5 (streaming) | the 30-client cell near the bottom: a `vectors:` line, and RAM that stays flat |

Baseline to compare against — the 2026-11-13 ResNet-50 run, 8 clients:

```
bootstrap: 33.4s/sample x 10 = ~330s
round 0 timing: phase1=1102.9s, phase2=257.7s, verify=256.9s, total=1617.5s
```

If any cell raises, stop and send me the full output and traceback — don't
skip ahead.

## Clone (idempotent — same pattern as 00_Setup, no `rmtree`)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# Same pattern as 00_Setup_and_Crypto_Tests.ipynb cell 2: only clone if
# it isn't already there. This is deliberately NOT `shutil.rmtree` then
# re-clone -- rmtree-ing a directory the kernel's cwd might already be
# inside breaks getcwd() for every subprocess (git, pip) spawned
# afterward in that kernel, which is exactly what happened the one time
# this was tried as a loose snippet instead of a real notebook.
if not os.path.exists('/content/FL-ZKP'):
    !git clone -q https://github.com/YashasweeNarayan/FL-ZKP.git /content/FL-ZKP
%cd /content/FL-ZKP
!git fetch -q origin main && git reset -q --hard origin/main
!git log --oneline -1

Expect the commit line to read `0bdd2da` or later. If it's older, this cell's
`git fetch && reset --hard` didn't run — check the cell above for errors
before continuing.

## Install / upgrade dependencies

In [ ]:
# rbcl (NOT pynacl) is what gives crypto/ristretto.py its ristretto255
# bindings. Mainline PyNaCl has never shipped ristretto255 support, under
# ANY version -- the PR adding it (github.com/pyca/pynacl/pull/646) is
# still open and unmerged upstream. An earlier attempt at this notebook
# chased the resulting AttributeError as a "stale pynacl, needs --upgrade"
# problem; that diagnosis was wrong (see FIX.md F8) -- no pynacl version
# fixes it. rbcl ships a manylinux2014 x86_64 wheel, so this is a plain
# install, no source build. pynacl itself stays installed too, since
# secagg.py's ECDH helper genuinely does use its (unrelated) X25519 support.
!pip install -q rbcl
!pip install -q pynacl
!pip install -q flwr-datasets[vision]
!pip install -q cryptography   # F2's AES-CTR mask PRG -- no slow fallback exists on purpose
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121

**If this is the first time this cell has run in this session, restart now:
Runtime -> Restart session.** Then re-run from the cell below (not from the
top -- Drive stays mounted and the repo stays cloned across a session
restart; only the Python process resets, which a fresh install can need).

In [ ]:
import sys
sys.path.insert(0, '/content/FL-ZKP')

import rbcl
print("rbcl OK, has ristretto255 bindings:", hasattr(rbcl, "crypto_core_ristretto255_from_hash"))
assert hasattr(rbcl, "crypto_core_ristretto255_from_hash"), (
    "rbcl imported but is missing crypto_core_ristretto255_from_hash -- "
    "if this fires, rbcl's packaging changed; see crypto/ristretto.py's "
    "import fallback (it also tries rbcl.rbcl) and FIX.md F8."
)

import nacl
print("pynacl version (used only for plain X25519, not ristretto255):", nacl.__version__)

import cryptography
print("cryptography version:", cryptography.__version__)

import torch
print("torch OK, CUDA available:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

import flwr_datasets
print("flwr_datasets OK:", flwr_datasets.__version__)

## Config, FEMNIST

In [ ]:
import yaml

with open('configs/mvp_3day.yaml') as f:
    CFG = yaml.safe_load(f)
print("models in config:", CFG['mvp_scope']['models'])   # resnet18 first, not resnet50

from fl.data_noniid import prepare_case2

femnist_cfg = CFG['datasets']['femnist']
case2_data = prepare_case2(seed=CFG['seed'], cache_dir=femnist_cfg['cache_dir'])
print("clients:", len(case2_data['client_train']))
print("unseen_test writers (headline metric source):", len(case2_data['split'].unseen_test_writers))

## Models and keys

In [ ]:
from fl.models import build_model, flatten_state_dict

# Only resnet18 for this verification run, to spend as little GPU quota
# as possible. Add 'mobilenet_v2' back (or just use CFG['mvp_scope']['models'])
# once this checks out.
MODELS_TO_RUN = ['resnet18']

models = {name: build_model(name, num_classes=62, in_channels=1) for name in MODELS_TO_RUN}
flat_weights = {}
for name, m in models.items():
    flat, _ = flatten_state_dict(m)
    flat_weights[name] = flat
    print(f"{name} model_dim = {flat.numel():,}")   # the real number, not my estimate

In [ ]:
from crypto.ristretto import Scalar, scalar_mul_base
from crypto.dkg import generate_contribution, finalize
from fl.round_loop import build_x25519_keys, precompute_pairwise_secrets

n_clients = len(case2_data['client_train'])
t = CFG['protocol']['threshold']
n_c = CFG['protocol']['n_key_holders']

client_sks = [Scalar.random() for _ in range(n_clients)]
ring = [scalar_mul_base(sk) for sk in client_sks]

x25519_sks, x25519_pks = build_x25519_keys(n_clients)
pairwise_secrets = precompute_pairwise_secrets(x25519_sks, x25519_pks)

key_holder_ids = list(range(1, n_c + 1))
dkg_contributions = [generate_contribution(h, t, key_holder_ids) for h in key_holder_ids]
PK_trace, key_holder_shares = finalize(dkg_contributions, key_holder_ids)
print(f"ring of {len(ring)} published, PK_trace established among {n_c} key holders (t={t})")

## Verification run — 8 clients, 2 rounds

**The one thing that must hold: every round line says `aggregate=EXACT`.**
If a round instead prints `AGGREGATE MISMATCH`, F1 is wrong and nothing
downstream is trustworthy — stop and send me the output.

At 8 clients and ResNet-18 the vectors fit inside the default 2 GiB budget,
so there will be **no** `vectors:` line here — that's expected, not a sign
F5 didn't load. The 30-client cell further down is what exercises the spill
path.

In [ ]:
import time, numpy as np
from fl.round_loop import run_training_loop, bootstrap_norms_from_model_dim, release_gpu_memory

SMOKE_ROUNDS = 2
SMOKE_CLIENTS = 8   # must be >= K_MIN (8) in fl/server.py or every round aborts regardless of verification outcome

results = {}
for model_name in MODELS_TO_RUN:
    print(f"\n==== {model_name} (verification: {SMOKE_ROUNDS} rounds, {SMOKE_CLIENTS} clients) ====")
    model = models[model_name]
    dim = flat_weights[model_name].numel()

    t0 = time.time()
    root_norms = bootstrap_norms_from_model_dim(
        model_dim=dim,
        sketch_dim=CFG['sketch']['m'],
        sketch_seed=f"bootstrap-{model_name}".encode(),
        quant_bits=16,
        rng=np.random.default_rng(CFG['seed']),
    )
    print(f"bootstrap total: {time.time()-t0:.1f}s  (was ~330s at resnet50, one sample at a time)")

    final_w_t, history = run_training_loop(
        model=model,
        model_dim=dim,
        client_datasets=case2_data['client_train'][:SMOKE_CLIENTS],
        ring=ring[:SMOKE_CLIENTS],
        client_sks=client_sks[:SMOKE_CLIENTS],
        pairwise_secrets={i: {j: pairwise_secrets[i][j] for j in range(SMOKE_CLIENTS) if j != i} for i in range(SMOKE_CLIENTS)},
        pk_trace=PK_trace,
        server_root_norms=root_norms,
        CFG=CFG,
        rounds=SMOKE_ROUNDS,
    )
    results[model_name] = {'final_w_t': final_w_t, 'history': history}
    release_gpu_memory()

print("\n--- per-round summary ---")
for model_name, r in results.items():
    for h in r['history']:
        print(model_name, h)

Also worth checking: with `aggregate=EXACT` holding, the round-1
overflow/NaN warning from earlier Colab runs should be gone, since the
weights are no longer destroyed in round 0 before round 1 trains from them.

## F5 memory proof — 30 clients

This is the exact client count that caused the 2026-11-12 crash. Only run
this if you have GPU time to spare; it's not needed to confirm F1/F2/F4,
just F5 specifically.

30 clients at ResNet-18 is ~5.0 GiB of client vectors under the old code —
most of the way to Colab's ~12.7 GB ceiling. Under the fix it should sit
near 2.5 GiB in RAM with the rest spilled to disk.

One real caveat, not a bug: at 30 clients the self-mask share threshold is
`(2*30)//3 + 1 = 21`. If more than 9 clients get rejected by verification,
the round aborts with "only N distinct self-mask shares, need 21" — correct
behavior, but it would show up here as an unmasking abort rather than a
clean round.

In [ ]:
import psutil

MEM_CLIENTS = 30
proc = psutil.Process(os.getpid())
print(f"RSS before: {proc.memory_info().rss / 2**30:.2f} GiB")

model_name = MODELS_TO_RUN[0]
dim = flat_weights[model_name].numel()
print(f"client vectors the OLD code would have pinned for the whole round: "
      f"{2 * MEM_CLIENTS * dim * 8 / 2**30:.1f} GiB")

root_norms = bootstrap_norms_from_model_dim(
    model_dim=dim, sketch_dim=CFG['sketch']['m'],
    sketch_seed=b"bootstrap-mem", quant_bits=16,
    rng=np.random.default_rng(CFG['seed']), verbose=False)

t0 = time.time()
_, mem_history = run_training_loop(
    model=models[model_name],
    model_dim=dim,
    client_datasets=case2_data['client_train'][:MEM_CLIENTS],
    ring=ring[:MEM_CLIENTS],
    client_sks=client_sks[:MEM_CLIENTS],
    pairwise_secrets={i: {j: pairwise_secrets[i][j] for j in range(MEM_CLIENTS) if j != i} for i in range(MEM_CLIENTS)},
    pk_trace=PK_trace,
    server_root_norms=root_norms,
    CFG=CFG,
    rounds=1,
)
print(f"\nRSS after: {proc.memory_info().rss / 2**30:.2f} GiB   (wall {time.time()-t0:.0f}s)")
print(mem_history)

What to look for: a `round 0 vectors: ... spilled to disk ...` line (the
store doing its job), `aggregate=EXACT`, and the RSS staying in the low
single digits instead of climbing past 10.

## If something fails

Send the full output and traceback. I most want to see, whichever cell broke:

1. the real `model_dim` printed above;
2. the bootstrap total and the `round N timing:` line;
3. whether `aggregate=EXACT` held on both rounds;
4. from the 30-client cell, the `vectors:` line and the before/after RSS.

Honest flag: `TorchSketchProjector.project_batch` (F4 on GPU) and the whole
`round_loop` vector-store wiring (F5) have never run anywhere torch is
actually installed until this notebook runs them. If one of those two
breaks, that's the most likely place.